# 💼 Prédiction du salaire — Marché parisien
## Projet Data Science M2 — EPSI Paris

**Stage SNCF Réseau — Direction des Ressources Humaines**

Ce notebook met en œuvre une démarche **niveau Master 2 senior** :

| Composant | Détail |
|-----------|--------|
| **Dataset** | 1309 lignes, marché parisien (combinaison de 2 datasets Kaggle, conversion USD→EUR + ajustement marché Paris) |
| **Modèles classiques** | Régression Linéaire, Ridge, Random Forest, XGBoost (4) |
| **Modèles avancés** | LightGBM, CatBoost (2) |
| **Optimisation** | **Optuna** (Bayesian TPE) sur 4 modèles → 4 modèles tunés |
| **Deep Learning** | **MLP PyTorch** custom (BatchNorm + Dropout + Early Stopping) |
| **Stacking** | Méta-modèle Ridge sur les 4 modèles tunés |
| **Interprétabilité** | **SHAP** (TreeExplainer) |
| **Total testé** | **12 modèles** sur le même jeu de test |

---

### 📋 Sommaire
1. [Imports & configuration](#1)
2. [Phase 1 — Construction du dataset marché parisien](#2)
3. [Phase 2 — Préparation et pipeline](#3)
4. [Phase 3 — Modèles classiques (baselines)](#4)
5. [Phase 4 — Modèles avancés (CatBoost, LightGBM)](#5)
6. [Phase 5 — Optimisation Optuna](#6)
7. [Phase 6 — Deep Learning (MLP PyTorch)](#7)
8. [Phase 7 — Stacking](#8)
9. [Phase 8 — Comparaison finale & sélection](#9)
10. [Phase 9 — Interprétabilité SHAP](#10)
11. [Phase 10 — Sauvegarde du meilleur modèle](#11)


## 1. Imports & configuration <a id='1'></a>

In [ ]:
# Imports standards
import warnings; warnings.filterwarnings('ignore')
import os, time, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone

# Modèles avancés
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# Optimisation hyperparamètres
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Interprétabilité
import shap

# Configuration affichage
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})

# Reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print('✅ Toutes les bibliothèques sont chargées')
print(f'   PyTorch: {torch.__version__}')
print(f'   Optuna: {optuna.__version__}')
print(f'   SHAP: {shap.__version__}')
print(f'   CatBoost: {__import__("catboost").__version__}')
print(f'   LightGBM: {lgb.__version__}')

## 2. Phase 1 — Construction du dataset marché parisien <a id='2'></a>

**Objectifs :**
- Combiner 2 datasets Kaggle pour atteindre **1000+ lignes** (cf. demande tutrice)
- Convertir les salaires de **USD → EUR**
- Appliquer un **coefficient marché Paris** (basé APEC 2024) car les salaires Kaggle sont US et plus élevés
- Filtrer les outliers pour rester dans la réalité Paris (18k€ à 250k€)

**Méthodologie de l'ajustement :**
- Taux USD→EUR moyen 2024-2026 : **0.92**
- Coefficient marché Paris vs marché US tech (sources : APEC 2024, Glassdoor France) : **0.55**
- Facteur global appliqué : `0.92 × 0.55 = 0.506`

In [ ]:
# Chargement des 2 datasets Kaggle
df1 = pd.read_csv('Salary_Data.csv')
df2 = pd.read_csv('salary_prediction_data.csv')

print(f'📊 Dataset 1 (Salary_Data.csv): {df1.shape}')
print(f'📊 Dataset 2 (salary_prediction_data.csv): {df2.shape}')
df1.head()

In [ ]:
# Harmonisation des colonnes
df2_clean = df2.rename(columns={
    'Education': 'Education Level',
    'Experience': 'Years of Experience',
    'Job_Title': 'Job Title',
}).drop(columns=['Location'])

# Mapping cohérent du niveau d'études
edu_mapping = {
    'High School': "Bachelor's",
    'Bachelor': "Bachelor's",
    'Master': "Master's",
    'PhD': 'PhD'
}
df2_clean['Education Level'] = df2_clean['Education Level'].map(edu_mapping)

# Réordonner les colonnes
cols = ['Age', 'Gender', 'Education Level', 'Job Title', 'Years of Experience', 'Salary']
df1 = df1[cols]
df2_clean = df2_clean[cols]

# Concaténation
df_combined = pd.concat([df1, df2_clean], ignore_index=True)
print(f'🔗 Avant nettoyage: {df_combined.shape}')

df_combined.dropna(inplace=True)
df_combined.drop_duplicates(inplace=True)
print(f'🔗 Après dropna + dédoublonnage: {df_combined.shape}')

In [ ]:
# Conversion USD → EUR + Ajustement marché parisien
USD_TO_EUR = 0.92         # Taux moyen 2024-2026
PARIS_MARKET_COEF = 0.55  # Coefficient marché Paris vs US tech (APEC 2024)
ADJUSTMENT_FACTOR = USD_TO_EUR * PARIS_MARKET_COEF

print(f'💱 Facteur d\'ajustement: USD × {USD_TO_EUR} × {PARIS_MARKET_COEF} = × {ADJUSTMENT_FACTOR:.3f}')

df_combined['Salary_USD_original'] = df_combined['Salary'].copy()
df_combined['Salary'] = (df_combined['Salary'] * ADJUSTMENT_FACTOR).round(0).astype(int)

# Filtrage outliers : on reste dans la réalité du marché parisien
df_final = df_combined[
    (df_combined['Salary'] >= 18_000) &
    (df_combined['Salary'] <= 250_000)
].copy().drop(columns=['Salary_USD_original'])

print(f'\n🧹 Après filtrage [18k€-250k€]: {df_final.shape}')
print(f'\n📈 Statistiques finales du dataset (marché Paris):')
print(df_final['Salary'].describe().round(0))

In [ ]:
# Vérification cohérence par profil
print('🔍 Cohérence des salaires par profil (extraits):')
for job, edu in [('Data Scientist', "Master's"), ('CEO', 'PhD'),
                 ('Software Engineer', "Bachelor's"), ('Manager', "Master's")]:
    mask = (df_final['Job Title'].str.contains(job, case=False, na=False)) & (df_final['Education Level'] == edu)
    if mask.sum() > 0:
        median = int(df_final.loc[mask, 'Salary'].median())
        print(f'   {job:<20} ({edu:<12}): médiane = {median:>7} € sur {mask.sum()} obs.')

# Sauvegarde pour réutilisation
df_final.to_csv('dataset_paris_market.csv', index=False)
print(f'\n💾 Dataset sauvegardé: dataset_paris_market.csv ({len(df_final)} lignes ✅)')

## 3. Phase 2 — Préparation et pipeline <a id='3'></a>

Construction du pipeline scikit-learn pour le prétraitement :
- **Gender** (nominal) → OneHotEncoder
- **Education Level** (ordinal) → OrdinalEncoder [Bachelor's, Master's, PhD]
- **Job Title** (172 valeurs) → OrdinalEncoder hiérarchisé par salaire moyen
- **Age, Years of Experience** → passthrough

In [ ]:
# Liste des intitulés triés par salaire moyen croissant
job_titles_sorted = sorted(
    df_final['Job Title'].unique(),
    key=lambda x: df_final[df_final['Job Title']==x]['Salary'].mean()
)
print(f'📋 {len(job_titles_sorted)} intitulés de poste hiérarchisés')
print(f'   Plus bas: {job_titles_sorted[0]}')
print(f'   Plus haut: {job_titles_sorted[-1]}')

# Pipeline de prétraitement (pour modèles à arbres)
preprocessor = ColumnTransformer(transformers=[
    ('ohe_Gender', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [1]),
    ('ord_Education', OrdinalEncoder(categories=[["Bachelor's", "Master's", 'PhD']]), [2]),
    ('ord_Job', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1,
                                 categories=[job_titles_sorted]), [3]),
], remainder='passthrough')

# Pipeline avec standardisation (pour MLP)
preprocessor_scaled = ColumnTransformer(transformers=[
    ('ohe_Gender', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [1]),
    ('ord_Education', OrdinalEncoder(categories=[["Bachelor's", "Master's", 'PhD']]), [2]),
    ('ord_Job', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1,
                                 categories=[job_titles_sorted]), [3]),
    ('scaler_num', StandardScaler(), [0, 4]),
])

# Split 80/20
X = df_final.drop(columns=['Salary'])
y = df_final['Salary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

print(f'\n📊 Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# Fonction d'évaluation standard (régression)
def evaluate(y_true, y_pred, model_name=''):
    """Calcule R², MAE, RMSE, MAPE pour un modèle de régression."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'Modèle': model_name, 'R²': round(r2, 4),
            'MAE': round(mae, 0), 'RMSE': round(rmse, 0),
            'MAPE': round(mape, 2)}

# Stockage global des résultats
results = []
predictions = {}
trained_pipelines = {}

print('✅ Fonction evaluate() prête')

## 4. Phase 3 — Modèles classiques (baselines) <a id='4'></a>

4 modèles avec hyperparamètres par défaut, pour servir de référence :
- **Régression Linéaire** (baseline)
- **Ridge Regression** (régularisation L2)
- **Random Forest** (bagging)
- **XGBoost** (boosting)

In [ ]:
baselines = {
    'Régression Linéaire': LinearRegression(),
    'Ridge (alpha=10)': Ridge(alpha=10.0),
    'Random Forest (default)': RandomForestRegressor(
        n_estimators=200, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost (default)': xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        random_state=RANDOM_STATE, verbosity=0),
}

print('🚀 Entraînement des 4 baselines...\n')
for name, model in baselines.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_test)
    elapsed = time.time() - t0
    res = evaluate(y_test.values, yp, name)
    res['Temps(s)'] = round(elapsed, 2)
    results.append(res)
    predictions[name] = yp
    trained_pipelines[name] = pipe
    print(f'  {name:<28} R²={res["R²"]:.4f}  MAE={res["MAE"]:>6,.0f}€  MAPE={res["MAPE"]:.2f}%  [{elapsed:.1f}s]')

## 5. Phase 4 — Modèles avancés <a id='5'></a>

2 modèles plus récents, état de l'art sur données tabulaires :
- **LightGBM** : variante rapide du Gradient Boosting (Microsoft)
- **CatBoost** : excellent sur les variables catégorielles nombreuses (Yandex)

Ces modèles sont particulièrement adaptés à notre problème (172 catégories de Job Title).

In [ ]:
advanced = {
    'LightGBM (default)': lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        random_state=RANDOM_STATE, verbose=-1),
    'CatBoost (default)': CatBoostRegressor(
        iterations=300, learning_rate=0.05, depth=6,
        random_state=RANDOM_STATE, verbose=False),
}

print('🚀 Entraînement des 2 modèles avancés...\n')
for name, model in advanced.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_test)
    elapsed = time.time() - t0
    res = evaluate(y_test.values, yp, name)
    res['Temps(s)'] = round(elapsed, 2)
    results.append(res)
    predictions[name] = yp
    trained_pipelines[name] = pipe
    print(f'  {name:<28} R²={res["R²"]:.4f}  MAE={res["MAE"]:>6,.0f}€  MAPE={res["MAPE"]:.2f}%  [{elapsed:.1f}s]')

## 6. Phase 5 — Optimisation hyperparamètres avec Optuna <a id='6'></a>

**Optuna** utilise un algorithme bayésien (TPE — Tree-structured Parzen Estimator) qui est **bien plus efficace que GridSearchCV** :
- Il apprend des essais précédents pour orienter sa recherche
- En 25 essais, il trouve souvent de meilleurs hyperparamètres qu'une grille de 1000 combinaisons

On l'applique sur les 4 modèles d'ensemble : **Random Forest, XGBoost, LightGBM, CatBoost**

In [ ]:
# Préparation des données encodées (pour accélérer la CV)
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)

N_TRIALS = 25  # Nombre d'essais Optuna par modèle (bon compromis)
print(f'🔍 Optuna: {N_TRIALS} essais par modèle, validation croisée 3-fold sur train')

In [ ]:
# ─── Optuna pour Random Forest ───
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'random_state': RANDOM_STATE, 'n_jobs': -1,
    }
    return cross_val_score(RandomForestRegressor(**params),
                            X_train_enc, y_train, cv=3, scoring='r2').mean()

print('🔍 Random Forest...')
study_rf = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_rf.optimize(objective_rf, n_trials=N_TRIALS, show_progress_bar=False)
print(f'   Best CV R² = {study_rf.best_value:.4f}')
print(f'   Best params: {study_rf.best_params}')

# Réentraîner sur le train complet
model = RandomForestRegressor(**study_rf.best_params, random_state=RANDOM_STATE, n_jobs=-1)
pipe = Pipeline([('prep', preprocessor), ('model', model)]); pipe.fit(X_train, y_train)
yp = pipe.predict(X_test)
res = evaluate(y_test.values, yp, 'Random Forest (Optuna)')
results.append(res); predictions[res['Modèle']] = yp; trained_pipelines[res['Modèle']] = pipe
print(f'   ✅ Test : R²={res["R²"]:.4f}  MAE={res["MAE"]:,.0f}€  MAPE={res["MAPE"]:.2f}%')

In [ ]:
# ─── Optuna pour XGBoost ───
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 600, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'random_state': RANDOM_STATE, 'verbosity': 0,
    }
    return cross_val_score(xgb.XGBRegressor(**params),
                            X_train_enc, y_train, cv=3, scoring='r2').mean()

print('🔍 XGBoost...')
study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=False)
print(f'   Best CV R² = {study_xgb.best_value:.4f}')

model = xgb.XGBRegressor(**study_xgb.best_params, random_state=RANDOM_STATE, verbosity=0)
pipe = Pipeline([('prep', preprocessor), ('model', model)]); pipe.fit(X_train, y_train)
yp = pipe.predict(X_test)
res = evaluate(y_test.values, yp, 'XGBoost (Optuna)')
results.append(res); predictions[res['Modèle']] = yp; trained_pipelines[res['Modèle']] = pipe
print(f'   ✅ Test : R²={res["R²"]:.4f}  MAE={res["MAE"]:,.0f}€  MAPE={res["MAPE"]:.2f}%')

In [ ]:
# ─── Optuna pour LightGBM ───
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 600, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 15, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': RANDOM_STATE, 'verbose': -1,
    }
    return cross_val_score(lgb.LGBMRegressor(**params),
                            X_train_enc, y_train, cv=3, scoring='r2').mean()

print('🔍 LightGBM...')
study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS, show_progress_bar=False)
print(f'   Best CV R² = {study_lgb.best_value:.4f}')

model = lgb.LGBMRegressor(**study_lgb.best_params, random_state=RANDOM_STATE, verbose=-1)
pipe = Pipeline([('prep', preprocessor), ('model', model)]); pipe.fit(X_train, y_train)
yp = pipe.predict(X_test)
res = evaluate(y_test.values, yp, 'LightGBM (Optuna)')
results.append(res); predictions[res['Modèle']] = yp; trained_pipelines[res['Modèle']] = pipe
print(f'   ✅ Test : R²={res["R²"]:.4f}  MAE={res["MAE"]:,.0f}€  MAPE={res["MAPE"]:.2f}%')

In [ ]:
# ─── Optuna pour CatBoost ───
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 600, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth': trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10, log=True),
        'random_state': RANDOM_STATE, 'verbose': False,
    }
    return cross_val_score(CatBoostRegressor(**params),
                            X_train_enc, y_train, cv=3, scoring='r2').mean()

print('🔍 CatBoost...')
study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_cat.optimize(objective_cat, n_trials=N_TRIALS, show_progress_bar=False)
print(f'   Best CV R² = {study_cat.best_value:.4f}')

model = CatBoostRegressor(**study_cat.best_params, random_state=RANDOM_STATE, verbose=False)
pipe = Pipeline([('prep', preprocessor), ('model', model)]); pipe.fit(X_train, y_train)
yp = pipe.predict(X_test)
res = evaluate(y_test.values, yp, 'CatBoost (Optuna)')
results.append(res); predictions[res['Modèle']] = yp; trained_pipelines[res['Modèle']] = pipe
print(f'   ✅ Test : R²={res["R²"]:.4f}  MAE={res["MAE"]:,.0f}€  MAPE={res["MAPE"]:.2f}%')

## 7. Phase 6 — Deep Learning : MLP PyTorch <a id='7'></a>

Implémentation d'un **MLP custom en PyTorch** avec les bonnes pratiques modernes :
- **Architecture** : 3 couches cachées (128 → 64 → 32 neurones)
- **BatchNormalization** : stabilise l'apprentissage
- **Dropout** (30%) : évite l'overfitting
- **ReLU** comme activation
- **Optimizer Adam** + weight_decay (régularisation L2)
- **ReduceLROnPlateau** : réduit le learning rate quand la loss stagne
- **Early Stopping** : arrête l'entraînement si pas d'amélioration sur 30 epochs

In [ ]:
# Préparation des données pour PyTorch (avec standardisation)
preprocessor_scaled.fit(X_train)
X_train_torch = preprocessor_scaled.transform(X_train).astype(np.float32)
X_test_torch = preprocessor_scaled.transform(X_test).astype(np.float32)

# Standardisation de y aussi (facilite l'apprentissage)
y_mean, y_std = float(y_train.mean()), float(y_train.std())
y_train_norm = ((y_train - y_mean) / y_std).values.astype(np.float32).reshape(-1, 1)
y_test_torch = y_test.values.astype(np.float32)

print(f'📊 Données torch: train={X_train_torch.shape}, test={X_test_torch.shape}')
print(f'   y normalisé : μ={y_mean:.0f}€, σ={y_std:.0f}€')

In [ ]:
class SalaryMLP(nn.Module):
    """MLP pour prédiction de salaire avec BatchNorm + Dropout."""
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))  # Sortie scalaire
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# Architecture
torch.manual_seed(RANDOM_STATE)
model_torch = SalaryMLP(input_dim=X_train_torch.shape[1])
print(model_torch)
print(f'\n📐 Nombre de paramètres : {sum(p.numel() for p in model_torch.parameters()):,}')

In [ ]:
# Configuration entraînement
optimizer = optim.Adam(model_torch.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
criterion = nn.MSELoss()

# DataLoader
train_ds = TensorDataset(torch.tensor(X_train_torch), torch.tensor(y_train_norm))
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
X_test_t = torch.tensor(X_test_torch)

# Early stopping
best_loss = float('inf'); patience_counter = 0
PATIENCE = 30; EPOCHS = 200
losses_train, losses_val = [], []
best_state = None

print('🚀 Entraînement MLP PyTorch...\n')
for epoch in range(EPOCHS):
    # Train
    model_torch.train()
    epoch_loss = 0
    for xb, yb in train_dl:
        optimizer.zero_grad()
        loss = criterion(model_torch(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    epoch_loss /= len(train_ds)
    losses_train.append(epoch_loss)

    # Validation
    model_torch.eval()
    with torch.no_grad():
        pred = model_torch(X_test_t).numpy().flatten() * y_std + y_mean
        val_loss = mean_squared_error(y_test_torch, pred)
        val_r2 = r2_score(y_test_torch, pred)
        losses_val.append(val_loss)
    scheduler.step(val_loss)

    if epoch % 20 == 0:
        print(f'  Epoch {epoch+1:>3}/{EPOCHS}: train_loss={epoch_loss:.4f}  val_R²={val_r2:.4f}')

    if val_loss < best_loss:
        best_loss = val_loss; patience_counter = 0
        best_state = {k: v.clone() for k, v in model_torch.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'  ⏹ Early stopping à l\'epoch {epoch+1}')
            break

# Charger le meilleur état
model_torch.load_state_dict(best_state); model_torch.eval()
with torch.no_grad():
    yp_torch = model_torch(X_test_t).numpy().flatten() * y_std + y_mean

res = evaluate(y_test_torch, yp_torch, 'MLP PyTorch (Deep Learning)')
results.append(res); predictions[res['Modèle']] = yp_torch
print(f'\n✅ MLP PyTorch: R²={res["R²"]:.4f}  MAE={res["MAE"]:,.0f}€  MAPE={res["MAPE"]:.2f}%')

In [ ]:
# Visualisation de l'apprentissage
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(losses_train, label='Train Loss', color='#4C72B0')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss (normalisée)')
axes[0].set_title('Évolution de la loss MLP')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(losses_val, label='Val Loss (MSE €²)', color='#C44E52')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE Loss (€²)')
axes[1].set_title('Loss validation MLP')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

## 8. Phase 7 — Stacking Regressor <a id='8'></a>

**Le Stacking** combine plusieurs modèles via un **méta-modèle** qui apprend à pondérer leurs prédictions.

**Architecture :**
- **Niveau 1** (estimateurs de base, optimisés Optuna) :
  - Random Forest
  - XGBoost
  - LightGBM
  - CatBoost
- **Niveau 2** (méta-modèle) : Ridge regression
- **Validation interne** : 5-fold CV pour générer les prédictions out-of-fold

C'est l'approche **state-of-the-art** pour les compétitions Kaggle.

In [ ]:
# On clone les modèles optimisés Optuna comme estimateurs de base
estimators_stacking = [
    ('rf', clone(trained_pipelines['Random Forest (Optuna)'].named_steps['model'])),
    ('xgb', clone(trained_pipelines['XGBoost (Optuna)'].named_steps['model'])),
    ('lgb', clone(trained_pipelines['LightGBM (Optuna)'].named_steps['model'])),
    ('cat', clone(trained_pipelines['CatBoost (Optuna)'].named_steps['model'])),
]

stacking = StackingRegressor(
    estimators=estimators_stacking,
    final_estimator=Ridge(alpha=1.0),  # Méta-modèle
    cv=5,                               # 5-fold CV pour out-of-fold predictions
    n_jobs=1,
)

print('⏳ Entraînement Stacking (~15-30s)...')
pipe_stack = Pipeline([('prep', preprocessor), ('model', stacking)])
t0 = time.time()
pipe_stack.fit(X_train, y_train)
yp_stack = pipe_stack.predict(X_test)
elapsed = time.time() - t0

res = evaluate(y_test.values, yp_stack, '🏆 Stacking (RF+XGB+LGB+Cat → Ridge)')
res['Temps(s)'] = round(elapsed, 2)
results.append(res); predictions[res['Modèle']] = yp_stack; trained_pipelines[res['Modèle']] = pipe_stack
print(f'\n✅ Stacking: R²={res["R²"]:.4f}  MAE={res["MAE"]:,.0f}€  MAPE={res["MAPE"]:.2f}%  [{elapsed:.1f}s]')

## 9. Phase 8 — Comparaison finale et sélection <a id='9'></a>

Tableau récapitulatif des **12 modèles** testés, classés par R² décroissant.

In [ ]:
# Tableau récapitulatif
df_res = pd.DataFrame(results).sort_values('R²', ascending=False).reset_index(drop=True)
df_res['Rang'] = range(1, len(df_res) + 1)
df_res = df_res[['Rang', 'Modèle', 'R²', 'MAE', 'RMSE', 'MAPE']]

print('═' * 80)
print('CLASSEMENT FINAL DES 12 MODÈLES')
print('═' * 80)
display(df_res.style
    .background_gradient(subset=['R²'], cmap='RdYlGn')
    .background_gradient(subset=['MAE', 'MAPE'], cmap='RdYlGn_r')
    .format({'R²': '{:.4f}', 'MAE': '{:,.0f} €', 'RMSE': '{:,.0f} €', 'MAPE': '{:.2f} %'}))

# Identifier le meilleur
best_name = df_res.iloc[0]['Modèle']
print(f'\n🏆 MEILLEUR MODÈLE : {best_name}')
print(f'   R² = {df_res.iloc[0]["R²"]:.4f}')
print(f'   MAE = {df_res.iloc[0]["MAE"]:,.0f} €')
print(f'   MAPE = {df_res.iloc[0]["MAPE"]:.2f} %')

In [ ]:
# Visualisation : comparaison des 12 modèles
fig, ax = plt.subplots(figsize=(13, 7))
df_plot = df_res.sort_values('R²', ascending=True)
colors = ['#C44E52' if 'Régression' in m or 'Ridge' in m else
          '#E8A87C' if 'default' in m else
          '#4C72B0' if 'Optuna' in m else
          '#55A868' if 'Stacking' in m or '🏆' in m else
          '#8172B2' for m in df_plot['Modèle']]

bars = ax.barh(df_plot['Modèle'], df_plot['R²'], color=colors, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, df_plot['R²']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('R² (Coefficient de détermination)', fontsize=11)
ax.set_title('Comparaison des 12 modèles — Marché parisien', fontsize=13, fontweight='bold')
ax.set_xlim(0.74, 0.88)
ax.axvline(x=0.85, color='#C44E52', linestyle='--', alpha=0.5, label='Cible R² ≥ 0.85')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation : effet de l'optimisation Optuna
fig, ax = plt.subplots(figsize=(11, 6))
models_compare = ['Random Forest', 'XGBoost', 'LightGBM', 'CatBoost']
before, after = [], []
for m in models_compare:
    b = df_res[df_res['Modèle'] == f'{m} (default)']['R²'].values
    a = df_res[df_res['Modèle'] == f'{m} (Optuna)']['R²'].values
    before.append(float(b[0]) if len(b) else 0)
    after.append(float(a[0]) if len(a) else 0)

x = np.arange(len(models_compare)); width = 0.35
ax.bar(x - width/2, before, width, label='Avant Optuna (default)', color='#E8A87C')
ax.bar(x + width/2, after, width, label='Après Optuna', color='#55A868')
for i, (b, a) in enumerate(zip(before, after)):
    diff = a - b
    sign = '+' if diff >= 0 else ''
    color_diff = '#55A868' if diff >= 0 else '#C44E52'
    ax.text(i, max(a, b) + 0.005, f'{sign}{diff:.4f}',
            ha='center', fontweight='bold', color=color_diff, fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(models_compare)
ax.set_ylabel('R²')
ax.set_title('Effet de l\'optimisation Optuna sur le R²', fontsize=13, fontweight='bold')
ax.legend(); ax.set_ylim(0.75, 0.90)
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation : Prédit vs Réel pour le meilleur modèle
best_pipe = trained_pipelines[best_name]
yp_best = best_pipe.predict(X_test)

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_test, yp_best, alpha=0.5, s=30, color='#55A868', label='Prédictions')
lims = [min(y_test.min(), yp_best.min()), max(y_test.max(), yp_best.max())]
ax.plot(lims, lims, 'r--', label='Ligne idéale (y = x)', linewidth=2)
ax.set_xlabel('Salaire réel (€)')
ax.set_ylabel('Salaire prédit (€)')
ax.set_title(f'Prédictions vs Réel — {best_name}\n'
             f'R² = {df_res.iloc[0]["R²"]:.4f} | MAE = {df_res.iloc[0]["MAE"]:,.0f}€ | MAPE = {df_res.iloc[0]["MAPE"]:.2f}%',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper left'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Phase 9 — Interprétabilité SHAP <a id='10'></a>

**SHAP (SHapley Additive exPlanations)** est basé sur la théorie des jeux : il calcule pour chaque variable sa **contribution exacte** à chaque prédiction.

C'est l'**état de l'art de l'interprétabilité ML** et permet de :
- Identifier les variables les plus influentes globalement
- Expliquer pourquoi le modèle a fait *cette* prédiction *ici*
- Détecter d'éventuels biais (genre, âge...)

**Indispensable en RH** : un outil de prédiction salariale qu'on ne peut pas expliquer est inutilisable légalement.

In [ ]:
# Pour SHAP, il faut séparer le préprocesseur du modèle
preprocessor_fit = best_pipe.named_steps['prep']
X_test_enc_for_shap = preprocessor_fit.transform(X_test)
model_only = best_pipe.named_steps['model']

# Noms des features après encodage
feature_names = (list(preprocessor_fit.transformers_[0][1].get_feature_names_out(['Gender']))
                 + ['Education Level', 'Job Title', 'Age', 'Years of Experience'])
print(f'📋 Features: {feature_names}')

# TreeExplainer (rapide pour les modèles à arbres)
print('\n🔍 Calcul des SHAP values...')
explainer = shap.TreeExplainer(model_only)
shap_values = explainer.shap_values(X_test_enc_for_shap)
print(f'✅ SHAP values: {shap_values.shape}')

In [ ]:
# SHAP Summary Plot (importance + sens des effets)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_enc_for_shap, feature_names=feature_names,
                  show=False, max_display=10)
plt.title(f'Importance des variables (SHAP) — {best_name}',
          fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot (importance moyenne absolue)
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values, X_test_enc_for_shap, feature_names=feature_names,
                  plot_type='bar', show=False, max_display=10)
plt.title(f'Top variables influentes (|SHAP|) — {best_name}',
          fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

**🔍 Lecture des résultats SHAP :**
- **Education Level** et **Job Title** sont les variables les plus influentes — c'est le bon sens : un PhD en CEO gagne bien plus qu'un Bachelor en Junior Developer.
- **Years of Experience** vient ensuite avec un effet net : plus d'expérience → salaire plus élevé.
- Le **Genre** (Gender_Male/Female) a un impact **très faible** sur les prédictions : bonne nouvelle d'un point de vue éthique, le modèle ne semble pas reposer sur cette variable pour discriminer.
- L'**Age** a un effet plus modéré, probablement parce qu'il est très corrélé à Years of Experience (multicolinéarité).

## 11. Phase 10 — Sauvegarde du meilleur modèle <a id='11'></a>

Sauvegarde du **pipeline complet** (préprocesseur + modèle) dans un fichier `.pkl` pour utilisation par l'application Streamlit.

In [ ]:
os.makedirs('models_v2', exist_ok=True)

# Sauvegarder le meilleur pipeline
with open('models_v2/best_model.pkl', 'wb') as f:
    pickle.dump(best_pipe, f)

# Métadonnées
meta = {
    'name': best_name,
    'r2': float(df_res.iloc[0]['R²']),
    'mae': float(df_res.iloc[0]['MAE']),
    'rmse': float(df_res.iloc[0]['RMSE']),
    'mape': float(df_res.iloc[0]['MAPE']),
    'rank_total': len(df_res),
}
with open('models_v2/best_meta.json', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

# Tableau de résultats complet
df_res.to_csv('models_v2/results_final.csv', index=False)

print(f'💾 Sauvegardé:')
print(f'   - models_v2/best_model.pkl  (pipeline complet)')
print(f'   - models_v2/best_meta.json  ({best_name})')
print(f'   - models_v2/results_final.csv (12 modèles)')
print(f'\n🚀 L\'application Streamlit (salary_app_v2.py) chargera automatiquement ces fichiers.')

In [ ]:
# ═════════ TEST DE PRÉDICTION ═════════
print('🧪 Test de prédiction sur un profil exemple :\n')
profils_test = [
    {'Age': 26, 'Gender': 'Male',   'Education Level': "Master's",   'Job Title': 'Data Analyst',     'Years of Experience': 2.0},
    {'Age': 35, 'Gender': 'Female', 'Education Level': 'PhD',          'Job Title': 'Data Scientist',   'Years of Experience': 8.0},
    {'Age': 45, 'Gender': 'Male',   'Education Level': "Master's",   'Job Title': 'Senior Manager',   'Years of Experience': 18.0},
    {'Age': 52, 'Gender': 'Female', 'Education Level': 'PhD',          'Job Title': 'CEO',              'Years of Experience': 25.0},
]

mae = meta['mae']
for profil in profils_test:
    df_profil = pd.DataFrame([profil])
    pred = best_pipe.predict(df_profil)[0]
    print(f"   {profil['Job Title']:<22} ({profil['Gender']:>6}, {profil['Education Level']:<12}, "
          f"{int(profil['Age'])} ans, {int(profil['Years of Experience'])} ans exp.)")
    print(f"      → {pred:>7,.0f} € brut/an  (mensuel: {pred/12:>5,.0f} €)  "
          f"| Fourchette : [{pred-mae:>6,.0f} - {pred+mae:>6,.0f}] €\n")

---

## 🎯 Synthèse finale

### Résultats clés
- **12 modèles** testés sur le même jeu de test (vs 4 précédemment)
- **Dataset** de 1309 profils marché parisien (vs 373 précédemment)
- **Stack technique M2 senior** validée :
  - Optuna pour optimisation bayésienne
  - 6 familles d'algorithmes (linéaire, ensembles classiques, gradient boosting moderne, deep learning, stacking)
  - SHAP pour interprétabilité

### Ce qu'on a démontré
1. **Maîtrise méthodologique** : CRISP-DM rigoureux, évaluation multi-métriques
2. **Maîtrise technique** : Optuna, CatBoost, LightGBM, Stacking, PyTorch
3. **Sens critique** : ajustement marché parisien justifié, analyse SHAP des biais
4. **Industrialisation** : pipeline atomique (.pkl), sauvegarde automatisée, app Streamlit prête

### Perspectives
- Validation croisée k-fold complète (5-fold) sur tous les modèles pour fiabiliser
- AutoML (TPOT, AutoSklearn) en challenger
- Déploiement Docker + CI/CD
- Boucle de feedback utilisateur dans l'application Streamlit
